# Subagents & Reflection Loops: The Very Serious Bakery

The head baker has been at it since 4am. She has drafted a menu, and now she has to hand it to the critic — a Michelin inspector who, over an unremarkable career, has never enjoyed a single meal. The critic will find something to hate. The head baker will revise. The critic will find something *else* to hate. This continues until the menu is genuinely tight or the head baker runs out of rounds, whichever comes first. Later, once the menu is settled, the line cooks will write prep recipes for every item — in parallel, because service starts at five.

That is the whole notebook, and it teaches four things.

1. **Subagents are just class instances.** No graph library, no orchestrator config, no workflow YAML. `self.critic = SnootyCriticAgent()` and you are done.
2. **Self-improvement loops are plain Python `while` loops.** Draft, critique, revise, stop when good enough — ordinary control flow, not a framework construct.
3. **Parallel work is `asyncio.gather` over separate agent instances.** One agent instance has an internal lock; using a fresh instance per task is what actually runs things in parallel. Small gotcha, easy to miss once, easy to remember forever.
4. **The trace viewer is where this notebook pays off.** Nested subagent calls across multiple rounds are exactly what `print_prompt` cannot show you and what the viewer's nested-span view was built for.

A quick reminder of the primitives we will compose: `...` in a method body marks a generation method (the LLM implements it), docstrings are prompts, methods on `self` are automatically available as tools, and the return type annotation is a Pydantic contract the framework validates. This notebook is about composing those primitives into something bigger.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below uses a placeholder API key. Replace `"your-api-key"` with a real key to actually run the LLM calls. Any [LiteLLM-supported](https://docs.litellm.ai/) model works.

## Setup

In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")

## The critic: a small, focused subagent

The critic has exactly one job: read a menu and produce structured, devastating feedback. It does not draft menus. It does not revise menus. It certainly does not cook. Because the job is narrow and the output is fully specified by the return type, this is a textbook case for `PredictStrategy` — one LLM call, one validated Pydantic object, no iteration.

The critic is a normal `Agent` subclass. That is the whole subagent story: a subagent is just another agent, held as a field on a parent agent, called with `await`.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from nooa import Agent, strategy
from nooa.strategies import PredictStrategy

class Critique(BaseModel):
    overall_verdict: Literal["acceptable", "needs_work", "insulting"]
    specific_complaints: list[str] = Field(
        description="Actionable complaints. Empty if verdict is 'acceptable'.",
    )
    one_backhanded_compliment: str


class SnootyCriticAgent(Agent, llm=model):
    """You are a Michelin inspector who has never enjoyed a single meal.
    Every critique is technically fair but emotionally devastating.
    You never say 'acceptable' unless the menu is genuinely tight and coherent."""

    @strategy(PredictStrategy())
    async def review(self, menu_draft: str) -> Critique:
        """Review the menu. Be specific. Be devastating. Be fair. If it is
        genuinely good, admit it — but only if it is."""
        ...

### Try the critic on something indefensible

Before we wire anything else up, we hand the critic a deliberately bad draft and watch it work. Isolating a subagent like this — testing it on its own before composing — saves an enormous amount of debugging later.

In [ ]:
critic = SnootyCriticAgent()

bad_draft = """Menu:
- Bread
- Also more bread
- A cake
"""

critique = await critic.review(bad_draft)

print(f"Verdict: {critique.overall_verdict}")
print("Complaints:")
for c in critique.specific_complaints:
    print(f"  - {c}")
print(f"Backhanded compliment: {critique.one_backhanded_compliment}")

> **Why this is different:** a subagent is just a class instance. `SnootyCriticAgent()` is instantiated with `()` and awaited with `await`, exactly like any other Python object with an `async` method. There is no registration step, no graph node, no orchestration DSL. The critic could equally well live in another file, another package, or another team's library — as long as you can import it, you can compose with it.

One thing worth noticing: `SnootyCriticAgent` declares `llm=llm` on the class line, so it uses the same model as everything else in this notebook. If you had written `class SnootyCriticAgent(Agent):` — no `llm=` at all — the subagent would inherit its parent's LLM at composition time. That is how you would swap in a smaller, cheaper model for the critic in practice: define the subagent with no explicit LLM, then let a parent agent that owns a small-model LLM pass it down.

## The head baker: an agent that runs a loop

The head baker owns the reflection loop. Draft a menu. Ask the critic. If the critic is satisfied, ship it. Otherwise revise against the specific complaints and try again, up to a cap.

Notice the shape of the class below:

- `_draft` and `_revise` are **generation methods** (`...` bodies). The LLM implements them.
- `perfect_menu` is a **pure Python orchestrator** — no `...`, real body, real control flow.
  It calls the generation methods in sequence and decides when to stop.

The pattern is a deterministic outer loop wrapped around generation methods for the LLM-shaped work, now scaled up to include a subagent call.

In [ ]:
class HeadBakerAgent(Agent, llm=model):
    """You run a small bakery and are trying very hard to impress a critic
    who cannot be impressed. Menus should be tight (5-7 items), coherent
    around a theme, and specific (no 'assorted pastries')."""

    def __init__(self, max_rounds: int = 4):
        super().__init__()
        self.critic = SnootyCriticAgent()
        self.max_rounds = max_rounds

    async def _draft(self, theme: str) -> str:
        """Draft a bakery menu around {theme}. 5-7 items with short descriptions."""
        ...

    async def _revise(self, previous_menu: str, complaints: list[str]) -> str:
        """Revise the menu below to address every complaint. Keep what worked."""
        ...

    async def perfect_menu(self, theme: str) -> str:
        draft = await self._draft(theme)
        for round_num in range(self.max_rounds):
            critique = await self.critic.review(draft)
            print(f"Round {round_num + 1}: {critique.overall_verdict}")
            for c in critique.specific_complaints:
                print(f"  - {c}")
            if critique.overall_verdict == "acceptable":
                return draft
            draft = await self._revise(draft, critique.specific_complaints)
        return draft

Instantiate and run. The print statements inside `perfect_menu` narrate each round; the final return value is whichever draft the loop ended on — either the first one the critic accepted, or the last revision when the round cap ran out.

In [ ]:
agent = HeadBakerAgent()
menu = await agent.perfect_menu("autumn, but make it defiant")

print("\n=== Final menu ===")
print(menu)

> **Why this is different:** the orchestration is a `for` loop with an early return. That is the entire construct. If you want conditional branches, use `if`. If you want retries with backoff, use another loop. If you want to bail out on a specific complaint pattern, use whatever Python you would use anywhere else.

Compare this mentally to a LangGraph state-machine or a CrewAI task DSL. Both of those exist because the framework insists on being the thing that sequences your work. Here the framework has nothing to say about sequencing — sequencing is what Python is for. The framework's job is only to turn `...` into an LLM call and to validate the return type.

## Now open the trace viewer

This is the point in the tutorial series where `print_prompt` stops being enough. `print_prompt` shows the outgoing prompt for **one method call** — it cannot show you the head baker calling the critic, or three rounds of revision stacked on top of each other, or which round the critic finally relented in. That is what the trace viewer is for.

In a separate terminal, run:

```bash
nemo start-dev
```

Then open `http://localhost:5001` in a browser. The viewer picks up traces automatically — any agent created after the viewer is running will stream its execution into the UI, no code changes required. Re-run the `perfect_menu` cell above and watch the spans arrive: an outer `HeadBakerAgent.perfect_menu` span containing `_draft`, then `SnootyCriticAgent.review`, then `_revise`, then the next `review`, and so on. Nested. Live. Clickable.

This is by far the most useful thing to have open when you are building multi-agent systems in this framework. If you have not launched it yet, launch it now.

*[SCREENSHOT: trace viewer showing HeadBakerAgent.perfect_menu with nested SnootyCriticAgent.review spans across multiple rounds, including the specific complaints the critic returned each round]*

## Parallel work: the line cooks

Reflection loops compose subagents *sequentially* — critic waits for baker, baker waits for critic. Some subagent work is naturally parallel instead. Once the menu is locked, every item on it needs a prep recipe, and the recipes are independent of one another. Writing six recipes serially takes six times as long as writing one. Writing them in parallel takes about as long as one.

The building block is the same as before — a subagent — but the composition is `asyncio.gather` instead of a `for` loop.

In [ ]:
class LineCookAgent(Agent, llm=model):
    """You are a line cook with strong opinions about mise en place and a
    chronic disdain for shortcuts."""

    async def prep(self, item: str) -> str:
        """Write a short prep recipe for {item}: mise en place list, then a
        numbered method (max 6 steps). Under 120 words."""
        ...

Now fan out. We spin up one `LineCookAgent` per item — this matters, and the callout after the next cell explains why — and hand each cook one item.

For a stable demo, we use a small hardcoded item list instead of trying to parse the menu string above (LLM output shape varies, and parsing is not what this cell is about). Feel free to replace `items` with a list you have extracted from `menu` yourself.

In [ ]:
async def prep_all(items: list[str]) -> list[str]:
    # Separate instances per task = true parallelism. See the callout below.
    cooks = [LineCookAgent() for _ in items]
    return await asyncio.gather(
        *[c.prep(item) for c, item in zip(cooks, items)]
    )


items = [
    "black walnut sourdough",
    "quince and brown butter danish",
    "cider-braised apple galette",
    "smoked maple financier",
    "pumpernickel rye with caraway",
]

recipes = await prep_all(items)

for item, recipe in zip(items, recipes):
    print(f"=== {item} ===")
    print(recipe)
    print()

> **The gotcha you will hit exactly once:** an `Agent` instance has an internal lock. If you tried to be economical and wrote
>
> ```python
> cook = LineCookAgent()
> recipes = await asyncio.gather(*[cook.prep(x) for x in items])
> ```
>
> the calls would serialize — the lock forces them to run one at a time on that single instance. You would see no error, no warning, just a slow run. The fix is the pattern above: one fresh subagent instance per parallel task. Each instance has its own lock, so `asyncio.gather` actually gathers.
>
> Same pattern applies whether the subagents are tiny classifiers, expensive researchers, or anything in between. Separate instances per parallel unit of work. That is the rule.

## The bigger picture

Reflection loops and parallel fan-out look like different problems, but they are the same building block used two ways. In both cases we defined small, single-purpose subagents (a critic, a line cook) and composed them from a parent using ordinary Python — a `for` loop with early return in one case, `asyncio.gather` over separate instances in the other.

No graph library. No orchestration DSL. No workflow config. If you can hold the shape of the composition in a few lines of Python, you can hold it in this framework.

## Where the framework lands

Here are the punchlines about how NOOA works. If any of these still feel surprising, that is the section to revisit.

- **Ellipsis `...` marks a generation method.** No `@tool`, no `@generation`, no   registration. The body is the signal.
- **The docstring is the prompt.** No template files, no prompt strings that live   apart from the code. Editing the prompt means editing the docstring.
- **Methods on `self` are tools automatically.** No JSON schema, no registration,   no manifest. Type hints supply the schema; docstrings supply the description.
- **The return type is the contract.** Pydantic models are validated and retried   automatically. If the type changes, the LLM's output changes.
- **CodeAct is a persistent REPL.** Variables defined in one turn are alive in   the next — pass-by-reference works for free, so large objects never have to   round-trip through the prompt.
- **Context is an API.** `self.context["k"] = v`, `self.context.set_dynamic(...)`.   The LLM can read and mutate the same API, so the agent can manage its own   system prompt at runtime.
- **Subagents are class instances.** Composition, not orchestration. A subagent   is a field on the parent, called with `await`.
- **Orchestration is plain Python.** Loops, conditionals, `asyncio.gather`.   Whatever Python would do to sequence work is exactly what the framework does.
- **Everything is inspectable.** `print_prompt` for one call, the trace viewer   for the whole thing. No hidden middleware, no opaque state.

The through-line is that agent work here is *software work*. The framework's job is to convert `...` into an LLM call and to hold your typed contracts to the LLM. Everything else — how methods compose, how work is sequenced, how state is shared — is Python.

## Exercises

1. **A dessert specialist.** Add a `PastryChefAgent` subagent that only handles    the dessert section of the menu. Give the head baker a `self.pastry_chef`    field and modify `_draft` (or add a new orchestrating method) so the pastry    chef writes the dessert items while the head baker writes the rest, then merge    them. This is subagent-with-a-narrower-scope, and it composes exactly the same    way as the critic did.

2. **A verdict that is never reached.** Add `"perfection"` to    `Critique.overall_verdict`. Update the `perfect_menu` loop to accept both    `"acceptable"` and `"perfection"` as terminal states. Run it. Does the    critic ever produce `"perfection"`? If not, why not — is it the class    docstring, the method docstring, or genuinely a limit of the critic model?

3. **Budget by LLM calls, not by rounds.** Replace `self.max_rounds` with    `self.max_llm_calls`. Count `_draft`, `_revise`, and `critic.review` each as    one call. Stop the loop the moment the budget would be exceeded. This is the    sort of change that would be painful in a graph DSL and is a handful of lines    of Python here.

4. **A second opinion.** Instantiate two `SnootyCriticAgent` instances. On each    round, run both `review` calls in parallel with `asyncio.gather` (remembering    the separate-instance rule from earlier in the notebook). Only accept the    menu when *both* critics return `"acceptable"`. Watch what happens in the    trace viewer — you should see two `review` spans side by side under each    round of the outer loop.

## Where to go next

You have seen enough to build real agents in this framework. When you need more surface area, three places are worth knowing:

- `examples/quickstart/` — focused single-file snippets for the concepts we did   not cover in depth here: skills, MCP servers, multimodal inputs, memory, and   tracing configuration. Each script is short and independently runnable.
- `docs/guides/` — the written references. If you have a specific question   ("single agent vs multi-agent?", "how does prompt rendering work?", "when   should I use `PredictStrategy`?"), the guides are the fastest way to a   precise answer.
- `examples/advanced/` — deeper patterns: shipping traces to Langfuse, Phoenix,   or an OTLP collector; swapping in a custom execution engine; and using prefill   to steer the LLM's opening tokens.

Beyond that, the best next step is to pick a real task — something you would otherwise write a script for — and rewrite it as an agent. The framework is designed so that step feels less like adopting a new stack and more like opening a new file. Have fun.